In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

IMD_RAW_DIR = ROOT / "data" / "raw" / "imd"

imd_files = sorted(
    list(IMD_RAW_DIR.glob("*.nc"))
    + list(IMD_RAW_DIR.glob("*.nc4"))
    + list(IMD_RAW_DIR.glob("*.cdf"))
)

print("Project root:", ROOT)
print("IMD directory:", IMD_RAW_DIR)
print("IMERG/IMD files found:", len(imd_files))

for index, file in enumerate(imd_files):
    print(
        f"[{index}] {file.name} "
        f"({file.stat().st_size / (1024 * 1024):.2f} MB)"
    )

if not imd_files:
    raise FileNotFoundError(
        "No IMD NetCDF file was found in data/raw/imd."
    )

IMD_FILE = imd_files[0]

with xr.open_dataset(IMD_FILE) as imd_source:
    print("\nIMD DATASET")
    print(imd_source)

    print("\nDimensions:")
    print(dict(imd_source.sizes))

    print("\nCoordinates:")
    for name, coordinate in imd_source.coords.items():
        print(
            name,
            coordinate.dims,
            coordinate.dtype,
            coordinate.shape,
        )

    print("\nData variables:")
    for name, variable in imd_source.data_vars.items():
        print(
            name,
            variable.dims,
            variable.dtype,
            variable.shape,
            variable.attrs.get("units"),
        )

    print("\nGlobal attributes:")
    for key, value in imd_source.attrs.items():
        print(f"{key}: {value}")

Project root: Z:\Projects\monsoon-postprocessing
IMD directory: Z:\Projects\monsoon-postprocessing\data\raw\imd
IMERG/IMD files found: 1
[0] RF25_ind2018_rfp25.nc (24.25 MB)

IMD DATASET
<xarray.Dataset> Size: 25MB
Dimensions:    (TIME: 365, LATITUDE: 129, LONGITUDE: 135)
Coordinates:
  * TIME       (TIME) datetime64[ns] 3kB 2018-01-01 2018-01-02 ... 2018-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Data variables:
    RAINFALL   (TIME, LATITUDE, LONGITUDE) float32 25MB ...
Attributes:
    history:      FERRET V6.82   20-Feb-26
    Conventions:  CF-1.0

Dimensions:
{'TIME': 365, 'LATITUDE': 129, 'LONGITUDE': 135}

Coordinates:
LONGITUDE ('LONGITUDE',) float64 (135,)
LATITUDE ('LATITUDE',) float64 (129,)
TIME ('TIME',) datetime64[ns] (365,)

Data variables:
RAINFALL ('TIME', 'LATITUDE', 'LONGITUDE') float32 (365, 129, 135) mm

Global attributes:
history: FERRET V6.82   20-Feb

In [2]:
CORRECTION_FILE = (
    ROOT / "data" / "processed" /
    "july2018_regime_correction_predictions.nc"
)

with xr.open_dataset(IMD_FILE) as imd_source:
    imd_rainfall = (
        imd_source["RAINFALL"]
        .rename(
            {
                "TIME": "date",
                "LATITUDE": "latitude",
                "LONGITUDE": "longitude",
            }
        )
        .load()
    )

imd_rainfall = imd_rainfall.astype("float32")
imd_rainfall.name = "imd_rainfall"

# Remove missing codes and physically invalid rainfall values.
imd_rainfall = imd_rainfall.where(
    (imd_rainfall >= 0)
    & (imd_rainfall <= 1000)
)

with xr.open_dataset(CORRECTION_FILE) as correction_source:
    target_dates = correction_source["date"].load()
    target_latitudes = correction_source["latitude"].load()
    target_longitudes = correction_source["longitude"].load()

# Match the July dates and existing model grid without spatial smoothing.
imd_aligned = imd_rainfall.reindex(
    date=target_dates,
    latitude=target_latitudes,
    longitude=target_longitudes,
)

imd_aligned.attrs.update(
    {
        "long_name": "IMD daily gridded rainfall",
        "units": "mm/day",
        "source": "India Meteorological Department",
        "original_resolution": "0.25 degree",
        "alignment_method": "exact coordinate reindexing",
    }
)

print(imd_aligned)

print("\nDate range:")
print(
    imd_aligned["date"].values[0],
    "to",
    imd_aligned["date"].values[-1],
)

print("\nCoordinate ranges:")
print(
    "Latitude:",
    float(imd_aligned.latitude.min()),
    "to",
    float(imd_aligned.latitude.max()),
)
print(
    "Longitude:",
    float(imd_aligned.longitude.min()),
    "to",
    float(imd_aligned.longitude.max()),
)

print("\nRainfall statistics:")
print("Minimum:", float(imd_aligned.min(skipna=True)))
print("Maximum:", float(imd_aligned.max(skipna=True)))
print("Mean:", float(imd_aligned.mean(skipna=True)))
print("Valid values:", int(imd_aligned.notnull().sum()))
print("Missing values:", int(imd_aligned.isnull().sum()))

<xarray.DataArray 'imd_rainfall' (date: 31, latitude: 129, longitude: 121)> Size: 2MB
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [n

In [3]:
with xr.open_dataset(CORRECTION_FILE) as correction_source:
    imerg_aligned = (
        correction_source["imerg_rainfall"]
        .load()
        .astype("float32")
    )

imerg_aligned.name = "imerg_rainfall"
imerg_aligned.attrs.update(
    {
        "long_name": "NASA GPM IMERG Final daily rainfall",
        "units": "mm/day",
    }
)

imerg_aligned, imd_aligned = xr.align(
    imerg_aligned,
    imd_aligned,
    join="exact",
)

common_valid_mask = (
    imerg_aligned.notnull()
    & imd_aligned.notnull()
)

observation_difference = (
    imerg_aligned - imd_aligned
).where(common_valid_mask)

absolute_observation_difference = (
    abs(observation_difference)
)

validation_ds = xr.Dataset(
    {
        "imerg_rainfall": imerg_aligned,
        "imd_rainfall": imd_aligned,
        "observation_difference": observation_difference,
        "absolute_observation_difference":
            absolute_observation_difference,
        "common_valid_mask": common_valid_mask,
    }
)

validation_ds["observation_difference"].attrs.update(
    {
        "long_name": "IMERG minus IMD rainfall",
        "units": "mm/day",
    }
)

validation_ds[
    "absolute_observation_difference"
].attrs.update(
    {
        "long_name": "Absolute IMERG and IMD rainfall difference",
        "units": "mm/day",
    }
)

validation_ds.attrs.update(
    {
        "title": "July 2018 IMERG and IMD rainfall validation dataset",
        "period": "2018-07-01 to 2018-07-31",
        "IMERG_source": "NASA GPM IMERG Final V07",
        "IMD_source": "IMD 0.25 degree daily gridded rainfall",
        "difference_definition": "IMERG minus IMD",
        "purpose": "Observation reliability analysis",
    }
)

print(validation_ds)

<xarray.Dataset> Size: 8MB
Dimensions:                          (date: 31, latitude: 129, longitude: 121)
Coordinates:
  * date                             (date) datetime64[ns] 248B 2018-07-01 .....
  * latitude                         (latitude) float64 1kB 6.0 6.25 ... 38.0
  * longitude                        (longitude) float64 968B 68.0 ... 98.0
Data variables:
    imerg_rainfall                   (date, latitude, longitude) float32 2MB ...
    imd_rainfall                     (date, latitude, longitude) float32 2MB ...
    observation_difference           (date, latitude, longitude) float32 2MB ...
    absolute_observation_difference  (date, latitude, longitude) float32 2MB ...
    common_valid_mask                (date, latitude, longitude) bool 484kB F...
Attributes:
    title:                  July 2018 IMERG and IMD rainfall validation dataset
    period:                 2018-07-01 to 2018-07-31
    IMERG_source:           NASA GPM IMERG Final V07
    IMD_source:            

In [4]:
valid_imerg = (
    validation_ds["imerg_rainfall"]
    .where(validation_ds["common_valid_mask"])
    .values
)

valid_imd = (
    validation_ds["imd_rainfall"]
    .where(validation_ds["common_valid_mask"])
    .values
)

valid_pairs = (
    np.isfinite(valid_imerg)
    & np.isfinite(valid_imd)
)

imerg_values = valid_imerg[valid_pairs]
imd_values = valid_imd[valid_pairs]

difference = imerg_values - imd_values

rmse = float(
    np.sqrt(
        np.mean(difference ** 2)
    )
)

mae = float(
    np.mean(
        np.abs(difference)
    )
)

bias = float(
    np.mean(difference)
)

correlation = float(
    np.corrcoef(
        imerg_values,
        imd_values,
    )[0, 1]
)

agreement_metrics = pd.DataFrame(
    [
        {
            "comparison": "IMERG versus IMD",
            "valid_pairs": len(difference),
            "RMSE": rmse,
            "MAE": mae,
            "Bias_IMERG_minus_IMD": bias,
            "Correlation": correlation,
            "IMERG_mean": float(np.mean(imerg_values)),
            "IMD_mean": float(np.mean(imd_values)),
            "IMERG_max": float(np.max(imerg_values)),
            "IMD_max": float(np.max(imd_values)),
        }
    ]
)

display(agreement_metrics)

daily_coverage = pd.DataFrame(
    {
        "date": validation_ds["date"].values,
        "valid_pairs": (
            validation_ds["common_valid_mask"]
            .sum(dim=["latitude", "longitude"])
            .values
        ),
        "imerg_mean_mm": (
            validation_ds["imerg_rainfall"]
            .where(validation_ds["common_valid_mask"])
            .mean(
                dim=["latitude", "longitude"],
                skipna=True,
            )
            .values
        ),
        "imd_mean_mm": (
            validation_ds["imd_rainfall"]
            .where(validation_ds["common_valid_mask"])
            .mean(
                dim=["latitude", "longitude"],
                skipna=True,
            )
            .values
        ),
        "mean_absolute_difference_mm": (
            validation_ds[
                "absolute_observation_difference"
            ]
            .mean(
                dim=["latitude", "longitude"],
                skipna=True,
            )
            .values
        ),
    }
)

display(daily_coverage)

,comparison,valid_pairs,RMSE,MAE,Bias_IMERG_minus_IMD,Correlation,IMERG_mean,IMD_mean,IMERG_max,IMD_max
0,IMERG versus IMD,153822,18.956995,9.872002,0.437859,0.39607,9.296186,8.858328,214.695023,534.24585


,date,valid_pairs,imerg_mean_mm,imd_mean_mm,mean_absolute_difference_mm
0,2018-07-01,4962,8.410054,6.091929,9.532894
1,2018-07-02,4962,9.610206,9.038514,9.789434
2,2018-07-03,4962,9.493993,10.307514,10.747268
3,2018-07-04,4962,8.304131,8.927484,9.013477
4,2018-07-05,4962,5.811616,7.479423,8.275135
5,2018-07-06,4962,7.093056,6.070940,7.331610
6,2018-07-07,4962,8.661446,7.425169,8.340826
7,2018-07-08,4962,8.750187,7.842623,8.393433
8,2018-07-09,4962,9.810590,7.456935,9.160666
9,2018-07-10,4962,9.928585,8.554787,10.344262


In [5]:
OUTPUT_FILE = (
    ROOT / "data" / "processed" /
    "july2018_imerg_imd_validation.nc"
)

METRICS_FILE = (
    ROOT / "data" / "processed" /
    "july2018_imerg_imd_metrics.csv"
)

DAILY_FILE = (
    ROOT / "data" / "processed" /
    "july2018_imerg_imd_daily_summary.csv"
)

# Store the mask compactly.
validation_to_save = validation_ds.copy()

validation_to_save["common_valid_mask"] = (
    validation_to_save["common_valid_mask"]
    .astype("int8")
)

encoding = {
    "imerg_rainfall": {
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,
    },
    "imd_rainfall": {
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,
    },
    "observation_difference": {
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,
    },
    "absolute_observation_difference": {
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,
    },
    "common_valid_mask": {
        "dtype": "int8",
        "zlib": True,
        "complevel": 4,
    },
}

validation_to_save.to_netcdf(
    OUTPUT_FILE,
    engine="netcdf4",
    encoding=encoding,
)

agreement_metrics.to_csv(
    METRICS_FILE,
    index=False,
)

daily_coverage.to_csv(
    DAILY_FILE,
    index=False,
)

print("Validation dataset saved:", OUTPUT_FILE.exists())
print("Dataset location:", OUTPUT_FILE)
print("Metrics saved:", METRICS_FILE.exists())
print("Daily summary saved:", DAILY_FILE.exists())

Validation dataset saved: True
Dataset location: Z:\Projects\monsoon-postprocessing\data\processed\july2018_imerg_imd_validation.nc
Metrics saved: True
Daily summary saved: True


In [6]:
with xr.open_dataset(OUTPUT_FILE) as saved_ds:
    print(saved_ds)

    required_variables = {
        "imerg_rainfall",
        "imd_rainfall",
        "observation_difference",
        "absolute_observation_difference",
        "common_valid_mask",
    }

    variables_available = required_variables.issubset(
        set(saved_ds.data_vars)
    )

    expected_shape = (
        31,
        129,
        121,
    )

    imd_shape_correct = (
        saved_ds["imd_rainfall"].shape
        == expected_shape
    )

    valid_pair_count = int(
        saved_ds["common_valid_mask"].sum()
    )

    print("\nVALIDATION")
    print("Required variables available:", variables_available)
    print("Expected IMD shape:", imd_shape_correct)
    print("Valid IMERG–IMD pairs:", valid_pair_count)
    print(
        "Saved date range:",
        saved_ds.date.values[0],
        "to",
        saved_ds.date.values[-1],
    )

    if (
        variables_available
        and imd_shape_correct
        and valid_pair_count > 0
    ):
        print("Stage 2 dataset validation passed.")
    else:
        print("Stage 2 dataset validation failed.")

<xarray.Dataset> Size: 8MB
Dimensions:                          (date: 31, latitude: 129, longitude: 121)
Coordinates:
  * date                             (date) datetime64[ns] 248B 2018-07-01 .....
  * latitude                         (latitude) float64 1kB 6.0 6.25 ... 38.0
  * longitude                        (longitude) float64 968B 68.0 ... 98.0
Data variables:
    imerg_rainfall                   (date, latitude, longitude) float32 2MB ...
    imd_rainfall                     (date, latitude, longitude) float32 2MB ...
    observation_difference           (date, latitude, longitude) float32 2MB ...
    absolute_observation_difference  (date, latitude, longitude) float32 2MB ...
    common_valid_mask                (date, latitude, longitude) int8 484kB ...
Attributes:
    title:                  July 2018 IMERG and IMD rainfall validation dataset
    period:                 2018-07-01 to 2018-07-31
    IMERG_source:           NASA GPM IMERG Final V07
    IMD_source:             